In [ ]:
!pip install numpy pandas matplotlib seaborn scipy scikit-learn statsmodels pingouin scikit_posthocs xgboost -q
print("pip 설치 완료!")

In [ ]:
!pip install -U imbalanced-learn

In [ ]:
# ============================================================
# 라이브러리 Import
# ============================================================

# 데이터 처리 및 분석
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from scipy.stats import skew, kurtosis
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import scikit_posthocs as sp

# 머신러닝
from sklearn.preprocessing import MinMaxScaler

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)



In [ ]:
# ============================================================
# 데이터 소개
# ============================================================

# - 공정(Process) 데이터
#     - Shot ID: 주조 샷 고유 식별자
#     - Injection Speed: 용탕 주입 속도 (m/s)
#     - Die Temperature: 금형 온도 (°C)
#     - Casting Pressure: 주조 압력 (bar)
#     - Cooling Time: 냉각 시간 (s)

# - 센서(Sensor) 데이터
#     - Mold Temp Sensor: 금형 내 센서 온도 (°C)
#     - Hydraulic Pressure: 유압 압력 (bar)
#     - Vibration Sensor: 진동값 (Hz)
#     - Flow Rate Sensor: 유량 (L/min)

# - 불량(Defects) 데이터
#     - Defect Type: 발생한 불량 유형 (미성형, 박리, 기공, 평탄, 개재물 등)
#     - Defect Status: 양품(0) / 불량(1) 여부

## 데이터 로드

#### header를 사용하여 0행과 1행을 머리글 행으로 하는 멀티컬럼 생성

In [ ]:
df = pd.read_csv("../../data/DieCasting_Quality_Raw_Data.csv", header=[0, 1])
df_origin = df.copy()
print("="*60)
print("데이터 로드 완료!")
print("="*60)
print(f"\nArticles: {df.shape}")

print("\n[Data Info]")
df.info()

In [ ]:
#컬럼명 앞뒤 공백 제거

df.columns = pd.MultiIndex.from_tuples(
    [(col[0].strip(), col[1].strip()) for col in df.columns]
)

print(df.columns.tolist())

## 샘플 데이터 확인


In [ ]:
print("\n" + "="*60)
print("샘플 데이터")
print("="*60)
display(df.head())

## 기초 통계량 확인

In [ ]:
print("\n" + "="*60)
print("기초 통계")
print("="*60)
display(df.drop(columns=[
        ('Process', 'id'),
        ('Process', 'Product_Type'),
        ('Process', 'Shot')
    ]).describe())

In [ ]:
df.info()

# 데이터 전처리

## 데이터 정제

### 중복 데이터 확인

In [ ]:
print("\n" + "="*60)
print("중복 데이터 확인")
print("="*60)

# 전체 행 중복 확인
print("\n[전체 행 기준 중복]")
print(f"diecasting 중복: {df.duplicated().sum():,}건")

## 결측치 처리

In [ ]:
print("\n" + "="*60)
print("결측치 확인")
print("="*60)

missing_df = pd.DataFrame({
    '결측수': df.isnull().sum(),
    '결측비율(%)': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_df = missing_df[missing_df['결측수'] > 0].sort_values('결측수', ascending=False)

if len(missing_df) > 0:
    print("\n[결측치 현황]")
    display(missing_df)
else:
    print("\n결측치 없음")

Factory_Temp 공장 온도 측정값 float64

Factory_Humidity 공장 습도 측정값 float64

### 히스토그램 및 기초통계량 확인 시, 평균보다는 중앙값으로 결측치를 채우는 게 좋다고 판단

In [ ]:
print("\n" + "="*60)
print("결측치 처리")
print("="*60)

cols = [
    'Factory_Temp',
    'Factory_Temp_Min',
    'Factory_Temp_Max',
    'Factory_Humidity',
    'Factory_Humidity_Min',
    'Factory_Humidity_Max'
]

for col in cols:
    df[('Sensor', col)] = df[('Sensor', col)].fillna(
        df[('Sensor', col)].median()
    )

df.isnull().sum().sum()
print("중앙값으로 처리 완료!")

In [ ]:
print("\n" + "="*60)
print("결측치 확인")
print("="*60)

missing_df = pd.DataFrame({
    '결측수': df.isnull().sum(),
    '결측비율(%)': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_df = missing_df[missing_df['결측수'] > 0].sort_values('결측수', ascending=False)

if len(missing_df) > 0:
    print("\n[결측치 현황]")
    display(missing_df)
else:
    print("\n결측치 없음")

## 불량 개수 확인하기

### Defects가 0이나 1이 아닌 경우(2, 3) -> 1로 처리

In [ ]:
defect_df = df['Defects']

# 1 초과 값이 하나라도 있는 컬럼
defect_over1 = defect_df.columns[(defect_df > 1).any()]
display(defect_over1)

# 1 초과 값을 모두 1로 변경
df['Defects'] = df['Defects'].clip(upper=1)

print("1 초과 값 개수:", (df['Defects'] > 1).sum().sum())

In [ ]:
defect = defect_df.columns[(defect_df == 1).any()]  # 한 번이라도 불량이 발생한 컬럼
display(defect_df[defect])

ones_count = (defect_df == 1).sum() # 해당 컬럼의 불량개수확인
ones_count = ones_count[ones_count > 0]
display(ones_count.sort_values(ascending=False))


Defects 변수 대부분 → 0이 압도적

즉, 데이터 불균형 (Imbalanced data)

0 : 정상
1 : 불량

## 불량 비율 분석하기
_______________________
현재 데이터 상태

각 행 = 제품 1개

Defects 안의 여러 컬럼 = 결함 유형

값 1 = 해당 결함 존재

👉 따라서 한 제품에 결함이 하나라도 있으면 불량

In [ ]:
# 제품 단위 불량 여부 (하나라도 1이면 불량)
df[('Target','is_defect')] = (df['Defects'].sum(axis=1) > 0).astype(int)

print(df[('Target','is_defect')].value_counts())

In [ ]:
print("\n" + "="*60)
print("전체 불량률 분석")
print("="*60)

total = len(df)
defect_cnt = df[('Target','is_defect')].sum()
defect_rate = defect_cnt / total * 100

print(f"총 생산 수량: {total}")
print(f"총 불량 수량: {defect_cnt}")
print(f"불량률: {defect_rate:.2f}%")

In [ ]:
# 시각화

counts = df[('Target','is_defect')].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12,4))

# ================================
# 파이차트
# ================================
axes[0].pie(counts,
            labels=['정상(0)','불량(1)'],
            autopct='%1.2f%%',
            colors=['lightblue','lightcoral'],
            startangle=90)
axes[0].set_title("전체 불량 비율")

# ================================
# 바차트
# ================================
bars = axes[1].bar(['정상','불량'],
                   counts,
                   color=['skyblue','indianred'],
                   edgecolor='black')

axes[1].set_title("불량 개수")
axes[1].grid(axis='y', alpha=0.3)

# 숫자 표시
for bar in bars:
    height = bar.get_height()
    axes[1].text(
        bar.get_x() + bar.get_width()/2,
        height,
        f'{int(height)}',
        ha='center',
        va='bottom',
        fontsize=10,
        #fontweight='bold'
    )

plt.tight_layout()
plt.show()

### 결함 유형별 불량 기여도
### 음.. 별로 안 중요한 분석 같기도

In [ ]:
defect_sum = df['Defects'].sum().sort_values(ascending=False)

contribution = defect_sum / defect_cnt * 100
contribution = contribution.sort_values(ascending=False)

print("결함 기여도 Top 10")
display(contribution.head(10))

# 이상치 탐색

## 이상치 탐색 결론부터
### --> 이상치 비율이 적고, 제조업 분석에서는 이상치가 발생 가능 범위 내에만 있다면 중요한 신호가 되므로 삭제하지 않는다. 
### + 트리 기반 모델 (랜덤포레스트, XGBoost 등)은 이상치에 비교적 강건함
| 이상치 비율 | 해석           |
| ------ | ------------ |
| <1%    | 거의 없음        |
| 1~5%   | 자연스러운 변동 가능  |
| 5~10%  | 데이터 품질 확인 필요 |
| >10%   | 데이터 문제 가능    |


## 히스토그램 생성

In [ ]:
plt.figure(figsize=(30,30))
plt.subplots_adjust(hspace=0.38)
# 각 변수의 막대그래프 개수
for index,value in enumerate(df):
 sub=plt.subplot(12,5,index+1)
 sub.hist(df[value],facecolor=(144/255,171/255,221/255),
	 	 	 linewidth=.3,edgecolor='black')
 plt.title(value)

## Product_Type 영향 확인해보기

In [ ]:
# Product_Type 데이터 개수 및 비율 확인
display(df['Process']['Product_Type'].value_counts())
display(df['Process']['Product_Type'].value_counts(normalize=True) * 100)

In [ ]:
# 1) 분석할 그룹(예: Process + Sensor)을 합치고, Product_Type은 따로 가져오기
X = pd.concat([df['Process'], df.get('Sensor', pd.DataFrame(index=df.index))], axis=1)
product = df['Process']['Product_Type']  # Product_Type 위치가 여기라고 가정

# 2) 시각화에 방해되는 컬럼 제외 (id, Product_Type 자체 등)
drop_cols = [c for c in ['id', 'Product_Type'] if c in X.columns]
X = X.drop(columns=drop_cols, errors='ignore')

# 3) long-form으로 변환
long_df = X.copy()
long_df['Product_Type'] = product
long_df = long_df.melt(id_vars='Product_Type', var_name='feature', value_name='value')

# 4) Facet boxplot
g = sns.catplot(
    data=long_df,
    x='Product_Type', y='value',
    col='feature', col_wrap=5,
    kind='box',
    height=2.6, aspect=1.0,
    sharey=False
)
g.set_titles("{col_name}")
for ax in g.axes.flatten():
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 두 그룹 간 중앙값이 꽤 다른 경우가 존재
#### 이는 공정 조건이 제품 타입에 따라 다름을 의미(제조 레시피가 다름..?)
- 예시) Melting_Furnace_Temp 를 보면 product1이 product2보다 높은 온도를 요구하는 것을 알 수 있음
- 예시) 같은 Velocity라도 product1에는 적합한 수치인 반면 product2에서는 너무 낮아서 불량이 발생하는 수치가 될 수도 있는 것!
#### product_type 별로 [모델을 분리]해서 분석하거나
#### product_type을 [feature로 사용]하는 전략이 필요할 듯


In [ ]:
# Cohen's d로 Product_Type이 바뀔 때 공정 설정이 가장 많이 변하는 변수 찾기(효과크기 비교하기)

print("\n" + "="*60)
print("Product_Type이 바뀔 때 공정 설정이 가장 많이 변하는 변수")
print("="*60)

X = pd.concat([df['Process'], df['Sensor']], axis=1)
ptype = df['Process']['Product_Type']

g1 = X[ptype == 1]
g2 = X[ptype == 2]

mean_diff = g2.mean() - g1.mean()

pooled_std = np.sqrt((g1.var() + g2.var()) / 2)

effect_size = (mean_diff / pooled_std).abs()

effect_rank = effect_size.sort_values(ascending=False)

display(effect_rank)

print("Min / Max 변수는 표준편차가 0이라서 NaN으로 나옴. ")
print("-> 값이 고정되어있는 변수라 정보량이 거의 없긴 함")

### 센서 데이터(Sensor 변수)의 이상치 탐색 및 처리

In [ ]:
sensor_df = df['Sensor']
Q1 = sensor_df.quantile(0.25)
Q3 = sensor_df.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_mask = (sensor_df < lower_bound) | (sensor_df > upper_bound)
outlier_count = outlier_mask.sum()
display(outlier_count.sort_values(ascending=False))

### 이상치가 불량 발생과 연관있는 값인가? -> 이 경우 제거하지 말고 이상치 플래그 변수 추가 하는 방법이 있음

In [ ]:
df['Defect_Status'] = df['Defects'].max(axis=1) #한 행에 불량이 하나라도 있으면 1
df['Defect_Status'] = df['Defect_Status'].map({
    0: 'Normal',
    1: 'Defective'
})

outlier_rows = outlier_mask.any(axis=1) #이상치가 하나라도 있으면 True
outlier_rows = outlier_rows.map({
    False : 'No_outlier',
    True : 'Outlier'
})

# 이상치 없는 행들(False) 중 정상/불량 비율 vs 이상치 있는 행들(True) 중 정상/불량 비율을 비교하는 표
result_table = pd.crosstab(
    outlier_rows,
    df['Defect_Status'],
    normalize='index'
)

display(result_table)

print("\n" + "="*60)
print("해석")
print("="*60)
print("이상치가 있을 경우 불량률이 22.18% -> 27.01%로 약 4.83%p 증가")
print("상대 증가율은 27.01/22.18 = 1.22로 약 22% 더 높은 것으로 확인됨") 
print("통계적으로 유의한지는 카이제곱 검정으로 확인 필요")

#### 이상치 여부와 불량 발생 간 관계가 통계적으로 유의한지 확인해보기 -카이제곱 검정

In [ ]:
print("\n" + "="*60)
print("카이제곱 검정 환경 세팅-비율에서 개수로")
print("="*60)


from scipy.stats import chi2_contingency

# 개수 테이블 (normalize 제거!)
contingency_table = pd.crosstab(
    outlier_rows,
    df['Defect_Status']
)

display(contingency_table)

In [ ]:
print("\n" + "="*60)
print("카이제곱 검정 결과")
print("="*60)

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"Chi-square statistic: {chi2:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Degrees of freedom: {dof}")
print("")

print("p-value가 0.03으로 0.05보다 작으므로, 이상치 여부와 불량 발생 사이에는 통계적으로 유의한 관계가 있다.")
print("Cramer's V로 효과 크기 확인해보기")

In [ ]:
print("\n" + "="*60)
print("Cramer's V 계산")
print("="*60)

n = contingency_table.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency_table.shape)-1)))

print(f"Cramer's V: {cramers_v:.4f}")

print("")
print("Cramer's V 값이 0.2이므로 약한 관계일 가능성이 높다.")
print("이상치라고 제거하기보다는 놔두는 것이 좋을 것으로 판단...")

## 공정 변수 이상치 탐색

In [ ]:
print("\n" + "="*60)
print("이상치 개수 by IQR")
print("="*60)

process_df = df['Process']
Q1 = process_df.quantile(0.25)
Q3 = process_df.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_mask = (process_df < lower_bound) | (process_df > upper_bound)
outlier_count = outlier_mask.sum()
display(outlier_count.sort_values(ascending=False))

In [ ]:
print("\n" + "="*60)
print("이상치 비율 : 보통 5% 이하면 삭제하지 않는 경우도 많음")
print("="*60)


outlier_ratio = outlier_count / len(process_df) * 100
display(outlier_ratio.sort_values(ascending=False))

In [ ]:
process_df['Velocity_2'].hist(bins=50)
plt.title("Velocity_2 Distribution")
plt.show()

print("\n" + "="*60)
print("Right-skewed distribution+극단값 존재")
print("그러나 극단치 비율이 2.44%로, ")
print("이는 제조 데이터 기준 굉장히 clean한 편이므로 제거할 필요는 없음")
print("="*60)

In [ ]:
print("\n" + "="*60)
print("이상치 개수 by Z-SCORE")
print("="*60)

z_scores = (process_df - process_df.mean()) / process_df.std()
z_outliers = np.abs(z_scores) > 3  # 3표준편차로 설정
z_outlier_count = z_outliers.sum()

display(z_outlier_count.sort_values(ascending=False))

In [ ]:
print("\n" + "="*60)
print("이상치 비율 by Z-SCORE")
print("="*60)

z_outlier_ratio = (z_outlier_count / len(process_df)) * 100

display(z_outlier_ratio.sort_values(ascending=False))

In [ ]:
print("\n" + "="*60)
print("IQR과 Z-SCORE의 비교 (Outlier Ratio %)")
print("="*60)

# 이상치 비율 계산
iqr_ratio = (outlier_count / len(process_df)) * 100
z_ratio = (z_outlier_count / len(process_df)) * 100

comparison = pd.DataFrame({
    "IQR (%)": iqr_ratio,
    "Z-Score (%)": z_ratio
})
comparison = comparison.round(3)

display(comparison.sort_values("IQR (%)", ascending=False))

print("\n")
print("Z-SCORE는 정규 분포를 가정하기 때문에 Skewed distribution에서 이상치를 예민하게 탐지할 가능성 있음")
print("현재 데이터 특성 상 IQR 기준이 더 적합한 것으로 판단")
print("-"*100)

## 공정 변수(Process)와 불량 발생(Defects) 간 관계 분석

In [ ]:
plt.figure(figsize=(14,10))

sns.heatmap(
    process_df.corr(),
    cmap="YlGnBu",
    annot=True,
    fmt=".2f"
)

plt.title("Process Correlation Heatmap")

plt.show()

### t-test로 어떤 공정변수가 불량유형과 관련 있는지 자세히 확인

In [ ]:

defect_df = df['Defects']

from scipy.stats import ttest_ind

results = []

for defect in defect_df.columns:
    
    for var in process_df.columns:
        
        normal = process_df[defect_df[defect] == 0][var]
        defected = process_df[defect_df[defect] == 1][var]
        
        # 샘플 수가 너무 적은 경우 제외
        if len(defected) > 10:
            
            stat, p = ttest_ind(normal, defected, equal_var=False)
            
            results.append({
                "Defect": defect,
                "Variable": var,
                "p_value": p,
                "mean_normal": normal.mean(),
                "mean_defect": defected.mean()
            })

ttest_results = pd.DataFrame(results)

ttest_results = ttest_results.sort_values("p_value")

display(ttest_results.head(20))

In [ ]:

print("\n" + "="*100)
print("여러 불량에서 반복적으로 유의한 변수 순위")
print("")
print("숫자 의미 : 총 몇 개의 불량 유형에서 정상 vs 불량 평균 차이가 통계적으로 유의한가")
print("(숫자가 클수록 핵심 공정 변수, 즉 다이캐스팅 핵심 공정 파라미터일 가능성 높음)")
print("="*100)

significant_vars = ttest_results[ttest_results["p_value"] < 0.05]
rank = significant_vars.groupby("Variable").size().sort_values(ascending=False)

display(rank)

In [ ]:
for defect in defect_df.columns:

    data = process_df.copy()
    data['Defect'] = defect_df[defect]

    data_long = data.melt(id_vars='Defect',
                          var_name='Variable',
                          value_name='Value')

    g = sns.catplot(
        data=data_long,
        x="Defect",
        y="Value",
        col="Variable",
        kind="box",
        col_wrap=5,
        height=3,
        sharey=False
    )

    g.fig.suptitle(f"{defect} vs Process Variables", y=1.02)
    plt.show()